# 🌿 AgroLens AI – Crop Disease Classification Training Notebook

Welcome to the **AgroLens AI** model training notebook! In this notebook, we train a deep learning model using **MobileNetV2 Transfer Learning** to identify crop leaf diseases from images.

### What this notebook does:
1. Verifies GPU availability in Google Colab.
2. Downloads the **PlantVillage** dataset (color leaf images) using `kagglehub`.
3. Filters the dataset to keep **Tomato**, **Potato**, and **Pepper bell** classes (all disease and healthy classes).
4. Splits the dataset into **Train (70%)**, **Validation (15%)**, and **Test (15%)** sets.
5. Applies data augmentation and MobileNetV2 preprocessing.
6. Trains a **MobileNetV2** classifier in two stages (Feature Extraction + Fine-Tuning).
7. Evaluates the model with accuracy, precision, recall, F1-score, and a confusion matrix.
8. Generates **Grad-CAM** explainability heatmaps on test leaf images.
9. Exports `agrolens_model.keras`, `class_names.json`, and `model_metrics.json` for our FastAPI backend.

> **Tip:** In Google Colab, go to **Runtime → Change runtime type** and select **T4 GPU** before running the cells.

## Step 1: Check GPU and TensorFlow Version
First, let's import the necessary libraries, print the TensorFlow version, and confirm that a GPU (such as an NVIDIA T4) is available for fast training.

In [ ]:
import os
import json
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score

# Set fixed random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU is available: {gpus}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("⚠️ No GPU detected! Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")

## Step 2: Download the PlantVillage Dataset using `kagglehub`

We use `kagglehub` to download the **PlantVillage** dataset directly into Google Colab.

### 🔑 How to get Kaggle access if login is required:
Most public datasets download automatically with `kagglehub`. However, if you are prompted to authenticate:
1. Log in to your account at [https://www.kaggle.com](https://www.kaggle.com).
2. Click on your profile picture (top-right) and go to **Settings**.
3. Scroll down to the **API** section and click **Create New Token**. This downloads a `kaggle.json` file containing your `username` and `key`.
4. In Google Colab, you can either:
   - Run `kagglehub.login()` and enter your Kaggle username and API key when prompted, **or**
   - Add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Google Colab **Secrets** (the 🔑 icon in the left sidebar).

In [ ]:
!pip install -q kagglehub opencv-python-headless

import kagglehub

# Download the PlantVillage dataset
print("Downloading PlantVillage dataset from Kaggle...")
dataset_download_path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print(f"✅ Dataset downloaded to: {dataset_download_path}")

## Step 3: Filter for Tomato, Potato, and Pepper Bell Classes
For the first version of **AgroLens AI**, we focus on three major crops grown in Sri Lanka:
- **Tomato** (all disease classes + healthy)
- **Potato** (all disease classes + healthy)
- **Pepper bell** (all disease classes + healthy)

Let's locate the color image folders, copy only the classes belonging to these three crops into `/content/agrolens_filtered_data`, and display the image counts per class.

In [ ]:
# Find the directory containing the color PlantVillage class folders
def find_color_dataset_dir(root_path):
    root = Path(root_path)
    # First look specifically for a 'color' folder
    for p in root.rglob("*"):
        if p.is_dir() and p.name.lower() == "color":
            return p
    # Fallback: find any directory containing Tomato/Potato/Pepper subfolders
    for p in root.rglob("*"):
        if p.is_dir():
            subdirs = [d.name.lower() for d in p.iterdir() if d.is_dir()]
            if any(name.startswith("tomato") for name in subdirs) and any(name.startswith("potato") for name in subdirs):
                return p
    return root

source_data_dir = find_color_dataset_dir(dataset_download_path)
print(f"Source color images directory: {source_data_dir}")

filtered_data_dir = Path("/content/agrolens_filtered_data")
if filtered_data_dir.exists():
    shutil.rmtree(filtered_data_dir)
filtered_data_dir.mkdir(parents=True, exist_ok=True)

target_prefixes = ("tomato", "potato", "pepper")
valid_extensions = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

class_counts = {}
for folder in sorted(source_data_dir.iterdir()):
    if folder.is_dir() and folder.name.lower().startswith(target_prefixes):
        # Clean folder name so it is consistent across filesystems
        dest_folder = filtered_data_dir / folder.name
        dest_folder.mkdir(parents=True, exist_ok=True)
        
        images = [f for f in folder.iterdir() if f.is_file() and f.suffix in valid_extensions]
        for img_path in images:
            shutil.copy2(img_path, dest_folder / img_path.name)
        
        class_counts[folder.name] = len(images)

print(f"\n✅ Selected {len(class_counts)} classes (Total images: {sum(class_counts.values())}):\n")
for cls_name, count in class_counts.items():
    print(f"  - {cls_name:<50} : {count:>5} images")

## Step 4: Split Dataset into Train (70%), Validation (15%), and Test (15%)
We split each class folder into three separate directories (`train`, `val`, and `test`) using our fixed random seed (`SEED = 42`), and then load them using `tf.keras.utils.image_dataset_from_directory`.

In [ ]:
split_base_dir = Path("/content/agrolens_split_data")
if split_base_dir.exists():
    shutil.rmtree(split_base_dir)

train_dir = split_base_dir / "train"
val_dir = split_base_dir / "val"
test_dir = split_base_dir / "test"

rng = random.Random(SEED)

for class_folder in sorted(filtered_data_dir.iterdir()):
    if not class_folder.is_dir():
        continue
    
    images = sorted([f for f in class_folder.iterdir() if f.is_file()])
    rng.shuffle(images)
    
    n_total = len(images)
    n_train = int(0.70 * n_total)
    n_val = int(0.15 * n_total)
    
    splits = {
        train_dir / class_folder.name: images[:n_train],
        val_dir / class_folder.name: images[n_train:n_train + n_val],
        test_dir / class_folder.name: images[n_train + n_val:]
    }
    
    for split_class_dir, split_files in splits.items():
        split_class_dir.mkdir(parents=True, exist_ok=True)
        for img_path in split_files:
            shutil.copy2(img_path, split_class_dir / img_path.name)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print("Loading Train Dataset:")
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True
)

print("\nLoading Validation Dataset:")
raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

print("\nLoading Test Dataset:")
raw_test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

class_names = raw_train_ds.class_names
num_classes = len(class_names)
print(f"\n✅ Total Classes ({num_classes}): {class_names}")

## Step 5: Data Augmentation and MobileNetV2 Preprocessing
To help our model generalize better to real-world photos taken by farmers in the field, we apply random flips, rotations, zooms, and contrast adjustments during training.

We also apply `tf.keras.applications.mobilenet_v2.preprocess_input`, which scales pixel values from `[0, 255]` to `[-1, 1]`.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
], name="data_augmentation")

AUTOTUNE = tf.data.AUTOTUNE

def prepare_train_ds(ds):
    def augment_and_preprocess(images, labels):
        images = data_augmentation(images, training=True)
        images = tf.clip_by_value(images, 0.0, 255.0)
        images = preprocess_input(images)
        return images, labels
    return ds.map(augment_and_preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

def prepare_eval_ds(ds):
    def preprocess_only(images, labels):
        images = preprocess_input(images)
        return images, labels
    return ds.map(preprocess_only, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

train_ds = prepare_train_ds(raw_train_ds)
val_ds = prepare_eval_ds(raw_val_ds)
test_ds = prepare_eval_ds(raw_test_ds)

print("✅ Data augmentation and MobileNetV2 preprocessing pipelines are ready!")

## Step 6: Build the MobileNetV2 Transfer Learning Model
We load **MobileNetV2** pre-trained on ImageNet without its top classification layer (`include_top=False`) and freeze its weights for the initial training phase.

Architecture:
- `MobileNetV2` (ImageNet weights, `include_top=False`, frozen)
- `GlobalAveragePooling2D`
- `Dropout(0.3)`
- `Dense(num_classes, activation="softmax")`

In [ ]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze all layers in the base MobileNetV2 model
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
x = layers.Dropout(0.3, name="top_dropout")(x)
outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

model = models.Model(inputs=base_model.input, outputs=outputs, name="AgroLens_MobileNetV2")

model.summary()

## Step 7: Stage 1 Training (Feature Extraction – 10 Epochs)
We compile the model with `Adam(learning_rate=1e-3)` and train the custom classification head for 10 epochs using `EarlyStopping` and `ModelCheckpoint`.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_stage1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath="agrolens_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

INITIAL_EPOCHS = 10
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=callbacks_stage1
)

## Step 8: Stage 2 Training (Fine-Tuning – Unfreeze Last 30 Layers)
Now we unfreeze the last 30 layers of `MobileNetV2` so the deeper convolutional filters can adapt specifically to leaf textures and disease spots. We recompile with a much smaller learning rate (`Adam(learning_rate=1e-5)`) and train for 10 more epochs.

In [ ]:
# Unfreeze the last 30 layers of MobileNetV2 (keep BatchNormalization frozen for stability)
for layer in base_model.layers[-30:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_stage2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath="agrolens_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

FINE_TUNE_EPOCHS = 10
total_epochs = len(history_stage1.history["accuracy"]) + FINE_TUNE_EPOCHS

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=len(history_stage1.history["accuracy"]),
    epochs=total_epochs,
    callbacks=callbacks_stage2
)

## Step 9: Plot Training and Validation Accuracy & Loss Curves
Let's combine the history from both training stages and plot the accuracy and loss curves.

In [ ]:
acc = history_stage1.history["accuracy"] + history_stage2.history["accuracy"]
val_acc = history_stage1.history["val_accuracy"] + history_stage2.history["val_accuracy"]
loss = history_stage1.history["loss"] + history_stage2.history["loss"]
val_loss = history_stage1.history["val_loss"] + history_stage2.history["val_loss"]

fine_tune_start = len(history_stage1.history["accuracy"]) - 1

plt.figure(figsize=(14, 5))

# Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(acc, label="Training Accuracy", marker="o")
plt.plot(val_acc, label="Validation Accuracy", marker="o")
plt.axvline(x=fine_tune_start, color="gray", linestyle="--", label="Start Fine-Tuning")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(loss, label="Training Loss", marker="o")
plt.plot(val_loss, label="Validation Loss", marker="o")
plt.axvline(x=fine_tune_start, color="gray", linestyle="--", label="Start Fine-Tuning")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Evaluate on the Test Set
Now let's evaluate our best model on the unseen **Test Set (15%)**. We compute:
- Overall Test Accuracy
- Classification Report (Precision, Recall, F1-Score per class)
- Confusion Matrix Heatmap

In [ ]:
# Load the best saved checkpoint
best_model = tf.keras.models.load_model("agrolens_model.keras")

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

test_accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")

print(f"✅ Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"✅ Precision     : {precision * 100:.2f}%")
print(f"✅ Recall        : {recall * 100:.2f}%")
print(f"✅ F1-Score      : {f1 * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Plot Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("AgroLens AI – Test Set Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Step 11: Explainable AI with Grad-CAM Heatmaps
**Grad-CAM (Gradient-weighted Class Activation Mapping)** highlights which regions of the leaf image the model focused on when making its prediction.

Below, we define:
1. `make_gradcam_heatmap(img_array, model, last_conv_layer_name="Conv_1")`
2. `overlay_gradcam_on_image(original_img, heatmap, alpha=0.4)`

Then we visualize 5 random test leaf images along with their true label, predicted label, confidence score, and Grad-CAM overlay.

In [ ]:
import cv2

def make_gradcam_heatmap(img_array, model, last_conv_layer_name="Conv_1", pred_index=None):
    """
    Computes a Grad-CAM heatmap (values in [0, 1]) for a preprocessed input image batch (1, 224, 224, 3).
    """
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array, training=False)
        if isinstance(preds, (list, tuple)):
            preds = preds[0]
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam_on_image(original_img, heatmap, alpha=0.4):
    """
    Overlays a 2D Grad-CAM heatmap onto an RGB image (uint8 [0, 255]).
    """
    img_uint8 = np.uint8(np.clip(original_img, 0, 255))
    heatmap_resized = cv2.resize(heatmap, (img_uint8.shape[1], img_uint8.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    jet_heatmap = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    jet_heatmap = cv2.cvtColor(jet_heatmap, cv2.COLOR_BGR2RGB)

    superimposed_img = cv2.addWeighted(img_uint8, 1.0 - alpha, jet_heatmap, alpha, 0)
    return superimposed_img

# Collect all test file paths and true labels for random sampling
all_test_samples = []
for label_idx, cls_name in enumerate(class_names):
    cls_test_dir = test_dir / cls_name
    for img_file in cls_test_dir.iterdir():
        if img_file.is_file():
            all_test_samples.append((img_file, label_idx))

sample_items = random.sample(all_test_samples, 5)

plt.figure(figsize=(15, 18))
for i, (img_path, true_idx) in enumerate(sample_items):
    # Load original image
    pil_img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    orig_array = tf.keras.utils.img_to_array(pil_img)
    input_batch = np.expand_dims(orig_array.copy(), axis=0)
    preprocessed_batch = preprocess_input(input_batch)

    # Predict
    preds = best_model.predict(preprocessed_batch, verbose=0)[0]
    pred_idx = int(np.argmax(preds))
    confidence = float(preds[pred_idx]) * 100.0

    # Generate Grad-CAM heatmap and overlay
    heatmap = make_gradcam_heatmap(preprocessed_batch, best_model, last_conv_layer_name="Conv_1")
    overlay_img = overlay_gradcam_on_image(orig_array, heatmap, alpha=0.4)

    # Original Leaf
    plt.subplot(5, 3, 3 * i + 1)
    plt.imshow(np.uint8(orig_array))
    plt.title(f"True: {class_names[true_idx]}", fontsize=10)
    plt.axis("off")

    # Heatmap
    plt.subplot(5, 3, 3 * i + 2)
    plt.imshow(heatmap, cmap="jet")
    plt.title("Grad-CAM Heatmap", fontsize=10)
    plt.axis("off")

    # Overlay
    plt.subplot(5, 3, 3 * i + 3)
    plt.imshow(overlay_img)
    plt.title(f"Pred: {class_names[pred_idx]}\nConfidence: {confidence:.2f}%", fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 12: Save Model, Class Names, and Metrics & Download ZIP
Finally, we save the three artifacts needed by our FastAPI backend:
1. `agrolens_model.keras` – Trained Keras model
2. `class_names.json` – Ordered list of class names matching the model's softmax output
3. `model_metrics.json` – Evaluation metrics on the test dataset

We bundle these into `agrolens_model_artifacts.zip` and trigger a browser download.

In [ ]:
# 1. Ensure best model is saved as agrolens_model.keras
best_model.save("agrolens_model.keras")

# 2. Save class_names.json
with open("class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2)

# 3. Save model_metrics.json
metrics_data = {
    "test_accuracy": round(float(test_accuracy), 4),
    "precision": round(float(precision), 4),
    "recall": round(float(recall), 4),
    "f1_score": round(float(f1), 4),
    "num_classes": num_classes,
    "image_size": [224, 224],
    "architecture": "MobileNetV2"
}

with open("model_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_data, f, indent=2)

# 4. Zip all three files together
zip_filename = "agrolens_model_artifacts.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("agrolens_model.keras")
    zf.write("class_names.json")
    zf.write("model_metrics.json")

print(f"✅ Created {zip_filename} containing:")
print("   - agrolens_model.keras")
print("   - class_names.json")
print("   - model_metrics.json")

# 5. Download in Google Colab
try:
    from google.colab import files
    files.download(zip_filename)
except ImportError:
    print("Not running inside Google Colab; artifacts are saved in the current working directory.")

## 📦 Where to Place the Downloaded Files in AgroLens AI

Once `agrolens_model_artifacts.zip` finishes downloading to your computer, extract the 3 files (`agrolens_model.keras`, `class_names.json`, and `model_metrics.json`) and copy them into **both** of the following project directories:

1. **`ML/models/`** *(for ML archiving, evaluation, and future retraining reference)*:
   - `ML/models/agrolens_model.keras`
   - `ML/models/class_names.json`
   - `ML/models/model_metrics.json`

2. **`Backend/app/ml_models/`** *(for the FastAPI backend inference and Grad-CAM service)*:
   - `Backend/app/ml_models/agrolens_model.keras`
   - `Backend/app/ml_models/class_names.json`
   - `Backend/app/ml_models/model_metrics.json`